# <center> **Machine Learning for Social Data Science** <center/>

## <center> **Problem Set 2** <center/>

### **Preprocessing**

In [ ]:
#Load in libraries
from docx import Document
import csv
import os
import pandas as pd

In [ ]:
folder_path = "C:/Users/Ryan Whitehead/Desktop/OneDrive/Documents/Levelling Up/Funds/Community Ownership Fund/Project Profiles"

data = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".docx"):
            path = os.path.join(root, file)
            doc = Document(path)
            full_text = []
            filename_without_docx = os.path.splitext(file)[0]
            project_number, project_name = filename_without_docx.split(" – ", 1)
            for para in doc.paragraphs:
                if para.text.strip():
                    full_text.append(para.text.strip())
            text = " ".join(full_text)
            data.append({
                "Project_number": project_number,
                "Project_name": project_name,
                "Round": os.path.basename(root),
                "Text": text})

In [ ]:
#Remove project number and – from text
for entry in data:
    project_prefix = f"{entry['Project_number']} – "
    if entry['Text'].startswith(project_prefix):
        entry['Text'] = entry['Text'][len(project_prefix):].strip()

In [ ]:
#Regex
import re

# 1. Create a regex pattern to identify URLs starting with https://
# 'https://' matches the literal string.
# '\S+' matches one or more non-whitespace characters (the rest of the URL).
url_pattern = r'https://\S+'

# 2. Create a regex pattern to identify access dates
# '\[' and '\]' are used to match the literal square bracket characters.
# 'Date Accessed:' matches the literal text.
# '.*?' non-greedily matches any characters (the date itself) up until the closing bracket.
date_pattern = r'\[Date Accessed:.*?\]'

# 3. Write a loop to clean the raw text for each entry in the dataset
for entry in data:
    # Check if 'Text' exists in the dictionary and is a string type just to be safe
    if 'Text' in entry and isinstance(entry['Text'], str):
        
        # Extract the current raw text
        current_text = entry['Text']
        
        # Use re.sub() to substitute the URL pattern with an empty string ('')
        text_without_urls = re.sub(url_pattern, '', current_text)
        
        # Use re.sub() to substitute the Date pattern with an empty string ('')
        text_without_dates = re.sub(date_pattern, '', text_without_urls)
        
        # Strip removes any extra whitespace left at the start or end of the string
        cleaned_text = text_without_dates.strip()
        
        # 4. Update the dictionary with the fully cleaned text
        entry['Text'] = cleaned_text

# Optional: Print the first few entries to verify it worked
#print(data[:5])


In [ ]:
#Convert to Pandas Df and save corpus as CSV file for future ease and use
df = pd.DataFrame(data)
df.to_csv("Levelling_Up_COF_Corpus.csv", index = False)

### **Topic Modelling**

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [ ]:
COF = pd.read_csv("Levelling_Up_COF_Corpus.csv")

In [ ]:
text = COF.Text.tolist()

In [ ]:
topic_model = BERTopic()

topics, probabilities = topic_model.fit_transform(text)

topic_info = topic_model.get_topic_info()

In [ ]:
print({len(topic_info)})

In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN
from nltk.corpus import stopwords

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

umap_model = UMAP(n_neighbors = 3, n_components = 3, min_dist = 0.05)

hdbscan_model = HDBSCAN(min_cluster_size = 80, min_samples = 40, gen_min_span_tree = True, prediction_data = True) 

In [ ]:
stopwords = list(stopwords.words('english')) + ['community ownership', 'building', 'levelling up']

vectoriser_model = CountVectorizer(ngram_range = (1, 2), stop_words = stopwords)
representation_model = KeyBERTInspired()

model = BERTopic(
    embedding_model = embedding_model,
    vectorizer_model = vectoriser_model,
    representation_model = representation_model,
    calculate_probabilities = True)

In [ ]:
topics, probs = model.fit_transform(text)

In [ ]:
topic_info = model.get_topic_info()

In [ ]:
model.visualize_barchart()

### **Topic Modelling - Towards Data Science**

##### **Load in Data**

In [ ]:
COF = pd.read_csv("../Data/Levelling_Up_COF_Corpus.csv")

In [ ]:
text = COF.Text.tolist()

#### **Embeddings**

In [ ]:
#Step 1 - Embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(text)

#### **Dimensionality Reducation with UMAP**

In [ ]:
#Step 2 - Dimensionality Reduction
import umap

umap_model = umap.UMAP(n_neighbors = 5, min_dist = 0.05, metric = "cosine", random_state = 42)

umap_embeddings = umap_model.fit_transform(embeddings)

#### **Clustering with HDBSCAN**

In [ ]:
#Condensed Trees - Minimum Cluster Size
import hdbscan
import matplotlib.pyplot as plt

save_folder = "../Raw Figures"
os.makedirs(save_folder, exist_ok = True)

min_cluster_sizes = [3, 5, 8, 10, 15, 20, 30, 40, 50]

for min_size in min_cluster_sizes:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size = min_size,
        min_samples = 1,
        metric = "euclidean",
        cluster_selection_method = "eom")
    clusterer.fit(umap_embeddings)
    plt.figure(figsize = (8, 6))
    clusterer.condensed_tree_.plot(select_clusters = True)
    plt.title(f"min_cluster_size = {min_size}")
    plt.savefig(os.path.join(save_folder, f"Condensed_tree_cluster_{min_size}.png"))

In [ ]:
#Scatterplot - Minimum Cluster Size

for min_size in min_cluster_sizes:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size = min_size,
        min_samples = 1,
        metric = "euclidean",
        cluster_selection_method = "eom")
    clusterer.fit(umap_embeddings)
    labels = clusterer.labels_
    plt.figure(figsize = (8, 6))
    plt.scatter(
        umap_embeddings[:, 0],
        umap_embeddings[:, 1],
        c = labels,
        cmap = 'tab20',
        s = 15,
        alpha = 0.7)
    plt.title(f"min_cluster_size = {min_size}")
    plt.savefig(os.path.join(save_folder, f"Scatter_plot_cluster_{min_size}.png"))

In [ ]:
#Condensed Tree - Minimum Sample Size
min_sample_size = [1, 2, 3, 5, 10, 20, 40, 50, 80]

for min_size in min_sample_size:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size = 20,
        min_samples = min_size,
        metric = "euclidean",
        cluster_selection_method = "eom")
    clusterer.fit(umap_embeddings)
    plt.figure(figsize = (8, 6))
    clusterer.condensed_tree_.plot(select_clusters = True)
    plt.title(f"min_samples = {min_size}")
    plt.savefig(os.path.join(save_folder, f"Condensed_tree_sample{min_size}.png"))

In [ ]:
#Scatter plot - Minimum Sample Size

for min_size in min_sample_size:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size = 20,
        min_samples = min_size,
        metric = "euclidean",
        cluster_selection_method = "eom")
    clusterer.fit(umap_embeddings)
    labels = clusterer.labels_
    plt.figure(figsize = (8, 6))
    plt.scatter(
        umap_embeddings[:, 0],
        umap_embeddings[:, 1],
        c = labels,
        cmap = 'tab20',
        s = 15,
        alpha = 0.7)
    plt.title(f"min_samples = {min_size}")
    plt.savefig(os.path.join(save_folder, f"scatter_plot_sample_{min_size}.png"))

In [ ]:
#Final Model

clustering_model = hdbscan.HDBSCAN(min_cluster_size = 30, 
                                min_samples = 3,
                                metric = "euclidean",
                                cluster_selection_method = "eom") 

#### **Vectorisation**

In [ ]:
#Step 4 - Vectoriser
vectoriser_model = CountVectorizer(ngram_range = (1, 2), min_df = 0.01, max_df = 0.8, stop_words = 'english')

#### **Representation Model**

In [ ]:
#Step 6 - Representation Model
representation_model = KeyBERTInspired()

#### **BERTopic Model**

In [ ]:
#Run Model
topic_model = BERTopic(
    embedding_model = embedding_model,
    umap_model = umap_model,
    hdbscan_model = clustering_model,
    vectorizer_model = vectoriser_model,
    representation_model = representation_model)

topics, probabilities = topic_model.fit_transform(text)

topic_info = topic_model.get_topic_info()

print(topic_info)

In [ ]:
#Visualise the model
topic_model.visualize_barchart()

#### **Validation**